# Chapter 7: Backpropagation — How Weights Update

Backpropagation answers: **"How much should each weight change to reduce the error?"**

It's the algorithm that makes neural networks trainable. Without it, we'd have to guess weight changes randomly.

## The 4-Step Training Loop

```
1. Forward pass   → input flows through network → prediction
2. Calculate loss  → how wrong was the prediction?
3. Backward pass   → trace error back through each layer (backpropagation)
4. Update weights  → nudge each weight to reduce error
```

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    """Given sigmoid OUTPUT, return its derivative."""
    return output * (1 - output)

# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

print("XOR Problem: the simplest non-linear pattern")
print("A single neuron CANNOT solve this (Ch 6).")
print("We need a hidden layer + backpropagation.")
print()
for inputs, target in zip(X, y):
    print(f"  {inputs} → {target[0]}")

---
## Setup: Network with Numbers

```
Input A ──w1──→ [H1] ──w5──┐
       ╲w3╱              ├──→ [Output]
       ╱w4╲              │
Input B ──w2──→ [H2] ──w6──┘
```

6 weights + 3 biases = 9 parameters to learn.

In [ ]:
# Simple weights for traceable math
w1, w2 = 0.5, 0.3    # → H1
w3, w4 = 0.7, 0.2    # → H2
w5, w6 = 0.4, 0.6    # → Output
lr = 0.5

print("Initial Weights:")
print(f"  Input→Hidden:  w1={w1}, w2={w2}, w3={w3}, w4={w4}")
print(f"  Hidden→Output: w5={w5}, w6={w6}")
print(f"  Learning rate:  {lr}")
print(f"  Biases: 0 (for simplicity)")

---
## Step 1: Forward Pass

Feed input `[1, 1]` (expected: 0) through the network.

Each neuron: `output = sigmoid(weighted_sum)`

In [ ]:
A, B = 1, 1
expected = 0

print("STEP 1: FORWARD PASS  [1, 1] → expected 0")
print("=" * 55)

# Hidden layer
H1_in = A * w1 + B * w2
H1_out = sigmoid(H1_in)
print(f"\n  H1_input  = A×w1 + B×w2 = {A}×{w1} + {B}×{w2} = {H1_in}")
print(f"  H1_output = sigmoid({H1_in}) = {H1_out:.4f}")

H2_in = A * w3 + B * w4
H2_out = sigmoid(H2_in)
print(f"\n  H2_input  = A×w3 + B×w4 = {A}×{w3} + {B}×{w4} = {H2_in}")
print(f"  H2_output = sigmoid({H2_in}) = {H2_out:.4f}")

# Output
O_in = H1_out * w5 + H2_out * w6
O_out = sigmoid(O_in)
print(f"\n  O_input  = H1×w5 + H2×w6 = {H1_out:.4f}×{w5} + {H2_out:.4f}×{w6} = {O_in:.4f}")
print(f"  O_output = sigmoid({O_in:.4f}) = {O_out:.4f}")

print(f"\n  Prediction: {O_out:.4f}  (wanted: {expected})")

---
## Step 2: Calculate Error

In [ ]:
error = expected - O_out

print("STEP 2: ERROR")
print("=" * 55)
print(f"\n  error = expected - predicted = {expected} - {O_out:.4f} = {error:.4f}")
print(f"\n  Negative → prediction too HIGH → need to push DOWN")

---
## Step 3: Backward Pass

This is backpropagation. We calculate **how much each weight contributed to the error**.

### The Chain Rule

To know how w5 affects the final error, we ask:
- How does the error change with the output? (error × sigmoid derivative)
- How does the output change with w5? (H1's value)

Multiply them together → that's w5's gradient.

For hidden layer weights (w1-w4), the chain is longer:
- error → output → hidden → weight

In [ ]:
print("STEP 3: BACKWARD PASS")
print("=" * 55)

# --- Output layer ---
print("\n─── Output Layer ───")
O_sens = sigmoid_derivative(O_out)
delta_O = error * O_sens
print(f"  Sensitivity = sigmoid_deriv({O_out:.4f}) = {O_out:.4f} × {1-O_out:.4f} = {O_sens:.4f}")
print(f"  δ_output = error × sensitivity = {error:.4f} × {O_sens:.4f} = {delta_O:.4f}")
print(f"  (This is: how much the output should change)")

dw5 = delta_O * H1_out
dw6 = delta_O * H2_out
print(f"\n  Δw5 = δ_output × H1 = {delta_O:.4f} × {H1_out:.4f} = {dw5:.4f}")
print(f"  Δw6 = δ_output × H2 = {delta_O:.4f} × {H2_out:.4f} = {dw6:.4f}")

# --- Hidden layer ---
print("\n─── Hidden Layer ───")
print("  Propagate error backward through w5, w6:")

error_H1 = delta_O * w5
H1_sens = sigmoid_derivative(H1_out)
delta_H1 = error_H1 * H1_sens
print(f"\n  H1: error = δ_output × w5 = {delta_O:.4f} × {w5} = {error_H1:.4f}")
print(f"      sensitivity = {H1_sens:.4f}")
print(f"      δ_H1 = {error_H1:.4f} × {H1_sens:.4f} = {delta_H1:.4f}")

error_H2 = delta_O * w6
H2_sens = sigmoid_derivative(H2_out)
delta_H2 = error_H2 * H2_sens
print(f"\n  H2: error = δ_output × w6 = {delta_O:.4f} × {w6} = {error_H2:.4f}")
print(f"      sensitivity = {H2_sens:.4f}")
print(f"      δ_H2 = {error_H2:.4f} × {H2_sens:.4f} = {delta_H2:.4f}")

dw1 = delta_H1 * A
dw2 = delta_H1 * B
dw3 = delta_H2 * A
dw4 = delta_H2 * B
print(f"\n  Δw1 = δ_H1 × A = {delta_H1:.4f} × {A} = {dw1:.4f}")
print(f"  Δw2 = δ_H1 × B = {delta_H1:.4f} × {B} = {dw2:.4f}")
print(f"  Δw3 = δ_H2 × A = {delta_H2:.4f} × {A} = {dw3:.4f}")
print(f"  Δw4 = δ_H2 × B = {delta_H2:.4f} × {B} = {dw4:.4f}")
print(f"\n  If input were 0, that weight gets Δ=0 (no blame).")

---
## Step 4: Update Weights

In [ ]:
print("STEP 4: UPDATE WEIGHTS")
print("=" * 55)
print(f"Formula: new = old + learning_rate × Δ")
print(f"Learning rate = {lr}\n")

print(f"  {'Weight':<6} {'Old':>8} {'Change':>10} {'New':>8} {'Dir':>4}")
print(f"  {'─'*6} {'─'*8} {'─'*10} {'─'*8} {'─'*4}")

updates = [("w1", w1, dw1), ("w2", w2, dw2), ("w3", w3, dw3),
           ("w4", w4, dw4), ("w5", w5, dw5), ("w6", w6, dw6)]

for name, old, delta in updates:
    change = lr * delta
    new = old + change
    arrow = "↓" if change < 0 else "↑"
    print(f"  {name:<6} {old:>8.4f} {change:>+10.4f} {new:>8.4f} {arrow:>4}")

print(f"\n  All decreased — output was too high, so push everything down.")

---
## Full Training: Watch XOR Get Solved

In [ ]:
# Full training with matrix operations
np.random.seed(42)
w_ih = np.random.uniform(-1, 1, (2, 2))
w_ho = np.random.uniform(-1, 1, (2, 1))
b_h = np.random.uniform(-1, 1, (1, 2))
b_o = np.random.uniform(-1, 1, (1, 1))
lr = 0.5

print("Training XOR: 10,000 epochs")
print(f"{'─'*65}\n")

milestones = [0, 1, 5, 10, 50, 100, 500, 1000, 2000, 5000, 9999]

for epoch in range(10000):
    # Forward
    hidden = sigmoid(X @ w_ih + b_h)
    output = sigmoid(hidden @ w_ho + b_o)
    
    # Error
    error = y - output
    loss = np.mean(error ** 2)
    
    # Backward
    d_out = error * sigmoid_derivative(output)
    d_hid = (d_out @ w_ho.T) * sigmoid_derivative(hidden)
    
    # Update
    w_ho += hidden.T @ d_out * lr
    w_ih += X.T @ d_hid * lr
    b_o += np.sum(d_out, axis=0, keepdims=True) * lr
    b_h += np.sum(d_hid, axis=0, keepdims=True) * lr
    
    if epoch in milestones:
        p = output.flatten()
        checks = "".join(["✓" if (v < 0.1 and t == 0) or (v > 0.9 and t == 1) else "✗" 
                          for v, t in zip(p, y.flatten())])
        print(f"  Epoch {epoch:>5} │ Loss: {loss:.6f} │ {p.round(3)} │ {checks}")

print(f"\n{'─'*65}")
print(f"\nFinal:")
print(f"  [0,0]→{output[0,0]:.4f}  (want 0) ✓")
print(f"  [0,1]→{output[1,0]:.4f}  (want 1) ✓")
print(f"  [1,0]→{output[2,0]:.4f}  (want 1) ✓")
print(f"  [1,1]→{output[3,0]:.4f}  (want 0) ✓")

---
## Batch vs Stochastic vs Mini-Batch

How many training examples do we use before updating weights?

| Method | Samples per Update | Pros | Cons |
|--------|-------------------|------|------|
| **Batch** | ALL | Stable, accurate gradient | Slow, needs lots of memory |
| **Stochastic (SGD)** | 1 | Fast, can escape local minima | Noisy, unstable |
| **Mini-Batch** | 32-256 | Best of both worlds | Most common in practice |

In [ ]:
# Demonstrate noise difference
print("Gradient Noise Comparison:\n")
print("Batch gradient descent (all samples):")
print("  Step 1: loss=0.50 → Step 2: loss=0.48 → Step 3: loss=0.46")
print("  Smooth, predictable decrease\n")

print("Stochastic gradient descent (1 sample):")
print("  Step 1: loss=0.50 → Step 2: loss=0.55 → Step 3: loss=0.42")
print("  Jumpy! But the general trend is still downward\n")

print("Mini-batch (32 samples):")
print("  Step 1: loss=0.50 → Step 2: loss=0.47 → Step 3: loss=0.45")
print("  Slightly noisy but mostly smooth — the practical default\n")

print("In practice: mini-batch of 32-256 samples is standard.")
print("GPUs are optimized for parallel math on batches.")

---
## The Vanishing Gradient Problem

Sigmoid derivative is at most 0.25. When you multiply small numbers through many layers:

```
Layer 5: gradient = 0.2
Layer 4: gradient = 0.2 × 0.2 = 0.04
Layer 3: gradient = 0.04 × 0.2 = 0.008
Layer 2: gradient = 0.008 × 0.2 = 0.0016
Layer 1: gradient = 0.0016 × 0.2 = 0.00032  ← almost zero!
```

Early layers barely learn. This is why **ReLU** replaced sigmoid in hidden layers — its derivative is 1 (no shrinking).

In [ ]:
# Demonstrate vanishing gradient
print("Vanishing Gradient: Why Sigmoid Fails in Deep Networks\n")
print(f"{'Layers':>7} {'Sigmoid grad':>14} {'ReLU grad':>12}")
print(f"{'─'*7} {'─'*14} {'─'*12}")

sig_grad = 1.0
relu_grad = 1.0
max_sig_deriv = 0.25  # maximum of sigmoid derivative

for layer in range(1, 11):
    sig_grad *= max_sig_deriv
    # ReLU derivative is 1 when active
    
    sig_bar = "█" * max(1, int(sig_grad * 100))
    relu_bar = "█" * 25
    
    print(f"{layer:>7} {sig_grad:>14.8f}  {sig_bar:<30} {'1.0':>8}  {relu_bar}")

print(f"\nAfter 10 layers with sigmoid: gradient is {sig_grad:.10f}")
print(f"After 10 layers with ReLU: gradient is still 1.0")
print(f"\nThis is why modern networks use ReLU for hidden layers.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Forward pass** | Input → weighted sums → activations → prediction |
| **Loss** | Measures prediction error (MSE, cross-entropy) |
| **Backward pass** | Chain rule traces error back to each weight |
| **Gradient** | Direction + magnitude of needed weight change |
| **Weight update** | `w -= lr × gradient` (step downhill) |
| **Mini-batch** | Update using 32-256 samples (practical standard) |
| **Vanishing gradient** | Sigmoid shrinks gradients; ReLU fixes this |

**Next chapter**: Deep Learning — what happens when we stack many layers